# Importing libraries

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("christinacdl/binary_hate_speech")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df['label'] = train_df['label'].apply(lambda x: 1 if x == 'OFF_HATEFUL_TOXIC' else 0)
val_df['label'] = val_df['label'].apply(lambda x: 1 if x == 'OFF_HATEFUL_TOXIC' else 0)
test_df['label'] = test_df['label'].apply(lambda x: 1 if x == 'OFF_HATEFUL_TOXIC' else 0)

train_df

,text,label
0,She won't be there for long.,0
1,i guess eu is gonna have to back track a littl...,0
2,@user @user @user @user @user I can understand...,1
3,Media Matters hates Joe diGenova - that's a re...,0
4,@user @user @user @user thanks to the best b'd...,0
...,...,...
31055,Actual animals however do object 😏,0
31056,&#8220;@PubesOnFleeK: My tweets trash&#8221;,1
31057,Confusing circumstances seem to get in the way...,0
31058,now new reading material for my #entrepreneuri...,0


# Dataset preprocessing

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [4]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        # 1. Embedding lookup
        embedded = self.embedding(input_ids)

        # 2. LSTM forward pass
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Extract the final hidden state
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # 4. Apply dropout
        hidden = self.dropout(hidden)  

        # 5. Final classification layer (returns logits)
        output = self.fc(hidden)

        return output

# Instancing the LSTM model, criterion and optimizer

In [6]:
embedding_dim = 128
hidden_dim = 128
output_dim = 1
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [8]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    """
    One epoch of training. 
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels' in each batch.
    - optimizer, criterion: training components (e.g., Adam, BCEWithLogitsLoss).
    - device: 'cpu' or 'cuda'.
    """
    model.train()
    losses = []
    correct_predictions = 0

    # For calculating precision, recall, F1:
    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # 1) Forward pass -> raw logits
        logits = model(input_ids)  # shape: (batch_size, 1)
        logits = logits.squeeze(dim=1)  # shape: (batch_size,)

        # 2) Compute loss (BCEWithLogitsLoss expects raw logits)
        loss = criterion(logits, labels.float())

        # 3) Backprop + optimization
        loss.backward()
        optimizer.step()

        # 4) Track loss
        losses.append(loss.item())

        # 5) Convert logits -> probabilities -> predicted classes
        probs = torch.sigmoid(logits)          # in [0, 1]
        preds_cls = (probs >= 0.5).long()      # threshold at 0.5

        # 6) Count correct predictions
        correct_predictions += torch.sum(preds_cls == labels)

        # 7) Collect for metric calculation
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds_cls.cpu().numpy())

    # Calculate overall metrics for the epoch
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1


def eval_model(model, data_loader, criterion, device):
    """
    Evaluation function. Similar to train_epoch, but no backprop.
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels'.
    - criterion: e.g., BCEWithLogitsLoss for binary classification.
    - device: 'cpu' or 'cuda'.
    """
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            # 1) Forward pass -> logits
            logits = model(input_ids)  # shape: (batch_size, 1)
            logits = logits.squeeze(dim=1)  # shape: (batch_size,)

            # 2) Compute loss
            loss = criterion(logits, labels.float())
            losses.append(loss.item())

            # 3) Convert logits -> probabilities -> predicted classes
            probs = torch.sigmoid(logits)
            preds_cls = (probs >= 0.5).long()

            correct_predictions += torch.sum(preds_cls == labels)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds_cls.cpu().numpy())

    # Metrics
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1

# Training loop

In [9]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device)
        
        val_acc, val_loss, val_prec, val_rec, val_f1 = eval_model(
            model, val_loader, criterion, device)
        
        print(f'Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, '
            f'Precision: {train_prec:.4f}, Recall: {train_rec:.4f}, F1 Score: {train_f1:.4f}')
        
        print(f'Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, '
            f'Precision: {val_prec:.4f}, Recall: {val_rec:.4f}, F1 Score: {val_f1:.4f}')
    return train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1

In [10]:
seeds = [2,3,5]
EPOCHS = 5
results = pd.DataFrame(columns=['seed', 'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1',
                                'val_loss', 'val_acc', 'val_prec', 'val_rec', 'val_f1',
                                'test_loss', 'test_acc', 'test_prec', 'test_rec', 'test_f1',
                                'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
                                'max_memory_usage_test', 'max_vram_usage_test', 'total_time_test'])

for seed in seeds:
    torch.manual_seed(seed)
    # Resetting model
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    )
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss().to(device)
    
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_train = perf_counter()
    max_memory_usage_train, retval = memory_usage((training_loop, (EPOCHS,), {}), retval=True, max_usage=True)
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1 = retval

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_test = perf_counter()
    max_memory_usage_test, retval = memory_usage((eval_model, (model, test_loader, criterion, device), {}), retval=True, max_usage=True)
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    test_acc, test_loss, test_prec, test_rec, test_f1 = retval

    results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
                                                val_loss, val_acc, val_prec, val_rec, val_f1,
                                                test_loss, test_acc, test_prec, test_rec, test_f1,
                                                max_memory_usage_train, max_vram_usage_train, total_time_train,
                                                max_memory_usage_test, max_vram_usage_test, total_time_test]],
                                                columns=results.columns)], ignore_index=True)
    

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6031, Accuracy: 0.6758, Precision: 0.6580, Recall: 0.7321, F1 Score: 0.6931
Val Loss: 0.5349, Accuracy: 0.7332, Precision: 0.6960, Recall: 0.8279, F1 Score: 0.7562
Epoch 2/5
Train Loss: 0.4925, Accuracy: 0.7673, Precision: 0.7446, Recall: 0.8138, F1 Score: 0.7776
Val Loss: 0.5034, Accuracy: 0.7615, Precision: 0.7422, Recall: 0.8011, F1 Score: 0.7706
Epoch 3/5
Train Loss: 0.4275, Accuracy: 0.8128, Precision: 0.7899, Recall: 0.8522, F1 Score: 0.8199
Val Loss: 0.4903, Accuracy: 0.7790, Precision: 0.7440, Recall: 0.8506, F1 Score: 0.7937
Epoch 4/5
Train Loss: 0.3626, Accuracy: 0.8518, Precision: 0.8283, Recall: 0.8877, F1 Score: 0.8570
Val Loss: 0.4937, Accuracy: 0.7837, Precision: 0.7762, Recall: 0.7970, F1 Score: 0.7865
Epoch 5/5
Train Loss: 0.3021, Accuracy: 0.8825, Precision: 0.8593, Recall: 0.9147, F1 Score: 0.8861
Val Loss: 0.5140, Accuracy: 0.7865, Precision: 0.7802, Recall: 0.7975, F1 Score: 0.7888


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_24216\3438238464.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6040, Accuracy: 0.6715, Precision: 0.6504, Recall: 0.7418, F1 Score: 0.6931
Val Loss: 0.5532, Accuracy: 0.7144, Precision: 0.6769, Recall: 0.8202, F1 Score: 0.7417
Epoch 2/5
Train Loss: 0.4933, Accuracy: 0.7627, Precision: 0.7406, Recall: 0.8085, F1 Score: 0.7731
Val Loss: 0.5013, Accuracy: 0.7595, Precision: 0.7465, Recall: 0.7857, F1 Score: 0.7656
Epoch 3/5
Train Loss: 0.4176, Accuracy: 0.8184, Precision: 0.7951, Recall: 0.8578, F1 Score: 0.8252
Val Loss: 0.4892, Accuracy: 0.7718, Precision: 0.7471, Recall: 0.8217, F1 Score: 0.7826
Epoch 4/5
Train Loss: 0.3499, Accuracy: 0.8557, Precision: 0.8308, Recall: 0.8934, F1 Score: 0.8610
Val Loss: 0.5047, Accuracy: 0.7785, Precision: 0.7423, Recall: 0.8532, F1 Score: 0.7939
Epoch 5/5
Train Loss: 0.2801, Accuracy: 0.8885, Precision: 0.8642, Recall: 0.9220, F1 Score: 0.8921
Val Loss: 0.5711, Accuracy: 0.7723, Precision: 0.7504, Recall: 0.8161, F1 Score: 0.7818


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6078, Accuracy: 0.6677, Precision: 0.6493, Recall: 0.7290, F1 Score: 0.6869
Val Loss: 0.5457, Accuracy: 0.7278, Precision: 0.6987, Recall: 0.8006, F1 Score: 0.7462
Epoch 2/5
Train Loss: 0.4959, Accuracy: 0.7603, Precision: 0.7409, Recall: 0.8008, F1 Score: 0.7696
Val Loss: 0.5088, Accuracy: 0.7474, Precision: 0.7603, Recall: 0.7223, F1 Score: 0.7408
Epoch 3/5
Train Loss: 0.4195, Accuracy: 0.8126, Precision: 0.7921, Recall: 0.8476, F1 Score: 0.8189
Val Loss: 0.5083, Accuracy: 0.7685, Precision: 0.7307, Recall: 0.8501, F1 Score: 0.7859
Epoch 4/5
Train Loss: 0.3500, Accuracy: 0.8561, Precision: 0.8305, Recall: 0.8948, F1 Score: 0.8614
Val Loss: 0.5245, Accuracy: 0.7690, Precision: 0.7260, Recall: 0.8640, F1 Score: 0.7890
Epoch 5/5
Train Loss: 0.2854, Accuracy: 0.8893, Precision: 0.8628, Recall: 0.9258, F1 Score: 0.8932
Val Loss: 0.5520, Accuracy: 0.7777, Precision: 0.7495, Recall: 0.8341, F1 Score: 0.7896


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [11]:
results.to_csv('results/lstm_binary1.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,val_loss,val_acc,val_prec,val_rec,...,test_acc,test_prec,test_rec,test_f1,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.302118,0.882453,0.859251,0.914746,0.886130,0.514018,0.786505,0.780242,0.797527,...,0.778264,0.770115,0.793512,0.781638,1224.917969,231.628906,85.797961,1223.378906,194.671875,1.624044
1,3,0.280091,0.888538,0.864196,0.921958,0.892143,0.571096,0.772341,0.750355,0.816074,...,0.767705,0.748092,0.807415,0.776622,1241.718750,232.615234,85.528049,1241.707031,194.754883,1.623922
2,5,0.285425,0.889311,0.862818,0.925821,0.893210,0.552046,0.777749,0.749537,0.834106,...,0.777234,0.747586,0.837281,0.789896,1226.914062,232.494141,85.791937,1226.890625,195.343750,1.607129


In [12]:
torch.save(model.state_dict(), 'results/lstm_binary1.pth')